![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 48 -- Competition: Twin Records

A railway company records every train trip twice. The **timetable log** knows the plan and the dispatchers' reports: how many stops the train has, how long the trip should take, how late it left and how late it arrived. A **GPS box** on the train only knows what it measured about once a minute: how long the train moved, how far it went, how often it stopped and how fast it was.

The two records of a trip used to share a number. A failed software update lost that number for every trip after 1 September, and shuffled the rows. One thing survived: the day and the hour each train left. So the records come in **blocks**: a block holds all the trips that left in one hour of one day, the same trips in both tables, in a different order and with different numbers.

**Your task:** for every September GPS record, find the log entry of the same trip.

**The files.** `train.csv`: 59,848 earlier trips (14 August to 1 September) in 435 blocks, with the log columns and the GPS columns side by side, so every pair is known. `test_a.csv`: the September log entries (`a_id`), 22,868 rows in 156 blocks. `test_b.csv`: the September GPS records (`b_id`), the same trips in the same blocks.

**Score.** Accuracy: the share of GPS records matched to the right log entry. Kaggle shows it on 40% of the GPS records during the competition and ranks you at the end on the other 60%.

**Rules of the task.** Use only NumPy, pandas, SciPy, scikit-learn and matplotlib, and only these three files.

**Your file.** `submission.csv` with the header `b_id,a_id` and 22,868 rows: every `b_id` of `test_b.csv` exactly once, next to the `a_id` you think is its pair.

**Dataset:** The setup cell leaves four files in `DATA_DIR`: `train.csv`, `test_a.csv`, `test_b.csv` and `sample_submission.csv` (a valid file: a random matching inside every block).

The clock started at the opening time, so start at once.

![Two records of one trip](https://raw.githubusercontent.com/KAUST-Academy/Olympiad2027/main/L1/Comps/November_Camp/Day48_Twin_Records/banner.png)

---
# The columns

**Timetable log** (`test_a.csv`, and the first half of `train.csv`):

| Column | Meaning |
|---|---|
| `block`, `hour` | the block (one day and hour) and the hour of departure |
| `sched_duration` | the planned travel time (minutes) |
| `n_stops` | the number of stations on the route, first and last included |
| `distance` | the straight-line distance from the first to the last station (km) |
| `mean_dwell` | the mean planned stop at a station (minutes) |
| `train_type` | 0 a local train, 1 a regional or fast train, 2 an InterCity |
| `departure_delay`, `arrival_delay` | how late the train left and arrived, as the dispatchers wrote it (minutes); these values are not always exact |
| `long_dwells` | the number of planned stops of 2 minutes or more |
| `min_leg`, `median_leg`, `max_leg` | the shortest, the middle and the longest planned time between two stations (minutes) |

**GPS box** (`test_b.csv`, and the second half of `train.csv`):

| Column | Meaning |
|---|---|
| `travel_time` | the time from the first to the last signal sent while moving (minutes) |
| `path_length` | the length of the path, from the signals (km) |
| `standstills` | how many times the train stood still |
| `stopped_share` | the share of the signals sent while standing still |
| `max_speed`, `mean_speed` | the highest and the mean speed while moving (m/s) |
| `speed_0` ... `speed_30_plus` | the share of the signals in each speed band (m/s) |
| `short_stops`, `long_stops` | the number of stops shorter than 2 minutes, and of 2 minutes or more |
| `turning` | the sum of the changes of direction per kilometre (degrees per km) |

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="tab-comp", student_id=STUDENT_ID)


In [ ]:
# DO NOT MODIFY: with this seed you get the same result on every re-run.
import random

import numpy as np
import pandas as pd

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/twin-records")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# Loading the files

`A_COLS` and `B_COLS` are the names of the log columns and of the GPS columns.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_a = pd.read_csv(os.path.join(DATA_DIR, "test_a.csv"))
test_b = pd.read_csv(os.path.join(DATA_DIR, "test_b.csv"))

A_COLS = [c for c in test_a.columns if c not in ("a_id", "block", "hour")]
B_COLS = [c for c in test_b.columns if c not in ("b_id", "block", "hour")]

print(f"train: {train.shape}, test_a: {test_a.shape}, test_b: {test_b.shape}")


In [ ]:
train.head()


In [ ]:
test_a.head()


In [ ]:
test_b.head()


---
# Validation

We split 40 blocks off the training set and shuffle them the same way as the test set: the log rows and the GPS rows get new, random identifiers. This way you can measure your solution in the same situation as the test. `val_key` holds the correct pairs.

In [ ]:
rng = np.random.default_rng(SEED)
val_blocks = rng.choice(train.block.unique(), 40, replace=False)

fit = train[~train.block.isin(val_blocks)].reset_index(drop=True)
val = train[train.block.isin(val_blocks)].reset_index(drop=True)

val["a_id"] = rng.permutation(len(val))
val["b_id"] = rng.permutation(len(val))

val_a = val[["a_id", "block", "hour"] + A_COLS].sample(frac=1, random_state=SEED).reset_index(drop=True)
val_b = val[["b_id", "block", "hour"] + B_COLS].sample(frac=1, random_state=SEED + 1).reset_index(drop=True)
val_key = val[["b_id", "a_id"]]


In [ ]:
def accuracy(pred, key):
    """Share of correctly matched b_ids."""
    m = key.merge(pred, on="b_id", suffixes=("", "_pred"), how="left")
    return (m.a_id == m.a_id_pred).mean()


---
# Baseline

Random matching within blocks: every GPS record gets a random log entry from its own block, and every log entry is used exactly once. This is a valid submission, but its accuracy is roughly 1 / (block size). Its score is the baseline: your points count from there upward.

In [ ]:
def random_match(a, b, seed=SEED):
    rng = np.random.default_rng(seed)
    out = []
    for block, bb in b.groupby("block"):
        aa = a[a.block == block]
        out.append(pd.DataFrame({"b_id": bb.b_id.values, "a_id": rng.permutation(aa.a_id.values)}))
    return pd.concat(out, ignore_index=True)


print(f"validation accuracy: {accuracy(random_match(val_a, val_b), val_key):.4f}")


The checks before saving: an upload with a wrong format also counts as an attempt, so only upload the file if the checks ran without errors.

In [ ]:
def save_submission(submission):
    submission = submission[["b_id", "a_id"]]

    assert len(submission) == len(test_b), f"expected {len(test_b)} rows, got {len(submission)}"
    assert submission.b_id.is_unique, "every b_id must appear exactly once"
    assert set(submission.b_id) == set(test_b.b_id), "the b_id values do not match the ids in test_b.csv"
    assert submission.a_id.isin(test_a.a_id).all(), "there is an a_id that does not appear in test_a.csv"
    assert submission.a_id.notna().all(), "every b_id needs an a_id"

    submission.astype("int64").to_csv("submission.csv", index=False)
    print("submission.csv saved:", submission.shape)


In [ ]:
# example submission: random matching within blocks
submission = random_match(test_a, test_b)
save_submission(submission)


In [ ]:
ck.submit("submission.csv")


---
# Hand in

**Every file goes to Kaggle.** `ck.submit` checks the format of `submission.csv` and records it; the score comes from Kaggle. On Colab, download the file first: open the Files panel (the folder icon on the left), find `submission.csv`, click the three dots beside it, then Download. On the competition's Kaggle page click **Submit Prediction**, choose the file and click **Submit**. Do the same, and run `ck.submit("submission.csv")` again, each time you write a better file.

**Your own cells go below**, as many as you need, above the Finish cell. End them with `save_submission(submission)`.

**Your final notebook** must run from top to bottom in under 20 minutes, and the file it writes last is the one you choose as your final submission on Kaggle.

In [ ]:
# Your code here


---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---

Made By **Sattam Altwaim**